# Trip showcase views

Run this **after** the pipeline notebook (`coober_pedy_trip_pipeline.py`). It builds the small labelled tables and views that the dashboard and the trip app read — nothing here recomputes the trip, it just re-packages the gold tables for display. Rerun it whenever you rebuild the pipeline.

Two ideas do all the work here:

* A **view** (`CREATE OR REPLACE VIEW`) is a saved *query*, not a copy of the data — so when the pipeline rebuilds the gold tables, every view below automatically shows the new numbers.
* A **table** (`CREATE OR REPLACE TABLE`) *does* copy data — used once only, for `day_labels`, because those labels are typed by a human and can't be derived from the watch files.

Each step below builds one view, followed by a quick preview query you can eyeball before moving on.

## Step 1: Day labels — the human layer

The watch records *where* and *when*, but never *what the day was*. This small hand-typed table adds that: one row per calendar day of the trip, with a day number and a short route description. Later, `dash_daily` joins it on `local_date` so charts and tables can show words as well as numbers.

Because the trip is fixed — 15 days, 10 to 24 Oct 2026 — the rows are simply listed in a `VALUES` clause: the easiest way in SQL to build a small table of known constants. `CREATE OR REPLACE TABLE` makes rerunning the cell safe — it drops the old copy and writes the new one.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.trip.day_labels AS
SELECT * FROM VALUES
  (DATE'2026-10-10', 1, 'Melbourne to the Penola area'),
  (DATE'2026-10-11', 2, 'Naracoorte Caves, Naracoorte and Mount Gambier'),
  (DATE'2026-10-12', 3, 'Naracoorte to Adelaide'),
  (DATE'2026-10-13', 4, 'Rest day in Adelaide'),
  (DATE'2026-10-14', 5, 'Adelaide to Roxby Downs'),
  (DATE'2026-10-15', 6, 'Roxby Downs to Andamooka and back'),
  (DATE'2026-10-16', 7, 'Roxby Downs to Coober Pedy'),
  (DATE'2026-10-17', 8, 'Coober Pedy sights and the Breakaways'),
  (DATE'2026-10-18', 9, 'Coober Pedy to William Creek'),
  (DATE'2026-10-19', 10, 'William Creek to Marree on the Oodnadatta Track'),
  (DATE'2026-10-20', 11, 'Marree to Beltana'),
  (DATE'2026-10-21', 12, 'Beltana, Hawker and Leigh Creek'),
  (DATE'2026-10-22', 13, 'Angorichina to Yunta and on towards Broken Hill'),
  (DATE'2026-10-23', 14, 'Into Broken Hill'),
  (DATE'2026-10-24', 15, 'Broken Hill to Melbourne')
AS t(local_date, day_number, route);

In [0]:
%sql
SELECT * FROM workspace.trip.day_labels ORDER BY day_number

## Step 2: Headline numbers (dash_kpis)

One row, one number per headline: total / recorded / estimated km, share recorded, moving hours, average moving speed, trip days vs **riding** days (`COUNT_IF(total_km > 0)` — of 15 calendar days, only the ones with kilometres count), longest day, top speed, total ascent.

Because everything is aggregated into a single row, this view can't be sliced by a date filter — by design. It's the fixed header of the trip, not a chart.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.trip.dash_kpis AS
SELECT
  ROUND(SUM(total_km), 0) AS total_km,
  ROUND(SUM(recorded_km), 0) AS recorded_km,
  ROUND(SUM(estimated_km), 0) AS estimated_km,
  ROUND(100 * SUM(recorded_km) / SUM(total_km), 0) AS pct_recorded,
  ROUND(SUM(recorded_moving_h) + SUM(estimated_moving_h), 1) AS moving_h,
  ROUND(SUM(recorded_km) / SUM(recorded_moving_h), 1) AS avg_moving_kmh,
  COUNT(*) AS trip_days,
  COUNT_IF(total_km > 0) AS riding_days,
  ROUND(MAX(total_km), 0) AS longest_day_km,
  ROUND(MAX(recorded_max_kmh), 1) AS top_speed_kmh,
  ROUND(SUM(recorded_ascent_m), 0) AS recorded_ascent_m
FROM workspace.trip.gold_daily;

In [0]:
%sql
SELECT * FROM workspace.trip.dash_kpis;

## Step 3: One row per day (dash_daily)

The dashboard's workhorse: `gold_daily` (the numbers) joined to `day_labels` (the words), plus a ready-made `day_label` like `Day 8 · 17 Oct` that charts can put straight onto an axis. The daily-kilometres chart and the date-range filter both read this view.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.trip.dash_daily AS
SELECT
  l.day_number,
  d.local_date,
  CONCAT('Day ', l.day_number, ' · ', date_format(d.local_date, 'd MMM')) AS day_label,
  l.route,
  d.total_km,
  d.recorded_km,
  d.estimated_km,
  d.pct_recorded,
  ROUND(d.recorded_moving_h + d.estimated_moving_h, 2) AS moving_h,
  d.recorded_moving_avg_kmh,
  d.recorded_max_kmh,
  d.recorded_ascent_m,
  d.curvature_deg_per_km,
  d.gap_ids
FROM workspace.trip.gold_daily d
JOIN workspace.trip.day_labels l USING (local_date);

In [0]:
%sql
SELECT day_label, route, total_km
FROM workspace.trip.dash_daily
ORDER BY total_km DESC
LIMIT 3;

## Step 4: Speed bands (dash_speed_bands)

How much riding time was spent at each speed. Two details worth noticing:

* Only **moving** points count (`is_moving` — speed above 10 km/h), so time at fuel stops and traffic lights doesn't distort the picture.
* `LEAST(step_s, 30)` caps every per-second step at 30 seconds: recordings occasionally jump (GPS wake-up, watch pauses), and without the cap one glitchy step would distort the total.

The cut points (40 / 60 / 80 / 100 / 110 km/h) are the same ones the route map's Speed Band colouring uses, so the map legend and this table always agree. The preview after the view also checks the share of time at 100 km/h and over.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.trip.dash_speed_bands AS
WITH banded AS (
  SELECT
    CASE
      WHEN speed_kmh < 40 THEN 1
      WHEN speed_kmh < 60 THEN 2
      WHEN speed_kmh < 80 THEN 3
      WHEN speed_kmh < 100 THEN 4
      WHEN speed_kmh < 110 THEN 5
      ELSE 6
    END AS band_order,
    LEAST(step_s, 30) AS seconds
  FROM workspace.trip.silver_track
  WHERE is_moving
)
SELECT
  band_order,
  CASE band_order
    WHEN 1 THEN '10 to 40 km/h' WHEN 2 THEN '40 to 60 km/h' WHEN 3 THEN '60 to 80 km/h'
    WHEN 4 THEN '80 to 100 km/h' WHEN 5 THEN '100 to 110 km/h' ELSE '110 km/h and over'
  END AS speed_band,
  ROUND(SUM(seconds) / 3600, 2) AS hours,
  ROUND(100 * SUM(seconds) / SUM(SUM(seconds)) OVER (), 1) AS pct_of_riding_time
FROM banded
GROUP BY band_order;

In [0]:
%sql
SELECT * FROM workspace.trip.dash_speed_bands ORDER BY band_order;

In [0]:
%sql
SELECT ROUND(SUM(pct_of_riding_time), 1) AS pct_100_plus
FROM workspace.trip.dash_speed_bands
WHERE band_order >= 5;

## Step 5: One point per kilometre (dash_route)

A map doesn't need 125,000 GPS points — one point per kilometre is plenty and draws far faster. This view buckets the whole track (recorded + estimated) with `FLOOR(trip_km)`, and inside each kilometre keeps:

* the position at the **start** of that km (`MIN_BY(lat, ts_utc)` — "the lat belonging to the earliest timestamp"), plus the earliest `local_date`;
* the **average** speed and altitude across the km — the same `speed_kmh` that feeds the map's Speed Band colour scale;
* whether **any** point in the km was imputed (`MAX(CAST(is_imputed AS INT)) = 1`), which drives the Recorded / Estimated colouring.

The first and last 5 km are trimmed (`BETWEEN 5 AND trip_total_km - 5`) so the map never shows exactly where home is.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.trip.dash_route AS
WITH points AS (
  SELECT *, MAX(trip_km) OVER () AS trip_total_km
  FROM workspace.trip.gold_track_filled
  WHERE lat IS NOT NULL
)
SELECT
  CAST(FLOOR(trip_km) AS INT) AS trip_km,
  MIN_BY(lat, ts_utc) AS lat,
  MIN_BY(lon, ts_utc) AS lon,
  MIN_BY(local_date, ts_utc) AS local_date,
  ROUND(AVG(speed_kmh), 1) AS speed_kmh,
  ROUND(AVG(altitude_m), 0) AS altitude_m,
  MAX(CAST(is_imputed AS INT)) = 1 AS is_imputed,
  CASE WHEN MAX(CAST(is_imputed AS INT)) = 1 THEN 'Estimated' ELSE 'Recorded' END AS source
FROM points
WHERE trip_km BETWEEN 5 AND trip_total_km - 5
GROUP BY CAST(FLOOR(trip_km) AS INT);

In [0]:
%sql
SELECT COUNT(*) AS points, MIN(trip_km) AS first_km, MAX(trip_km) AS last_km FROM workspace.trip.dash_route;

In [0]:
%sql
SELECT source, COUNT(*) AS points
FROM workspace.trip.dash_route
GROUP BY source;

## Step 6: Trip records (dash_records)

The superlatives: longest day, top speed, highest and lowest point, furthest north, twistiest day, biggest climb, slowest average, most estimated day. Almost all of it is **recorded** data — "Most estimated day" is the one row that leans on the estimated kilometres.

The function to learn here is `MAX_BY(x, y)`: it returns the `x` that goes with the biggest `y`. `MAX_BY(local_date, recorded_ascent_m)` is "the date of the biggest climb" in a single line — no subquery needed. `sort_order` keeps the rows in a fixed order, because the dashboard table sorts by it; left alone, a chart would sort the labels alphabetically.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.trip.dash_records AS
WITH track AS (
  SELECT * FROM workspace.trip.silver_track WHERE has_gps
),
days AS (
  SELECT * FROM workspace.trip.dash_daily
)
SELECT 1 AS sort_order, 'Longest day' AS record, CAST(MAX(total_km) AS DOUBLE) AS value, 'km' AS unit,
  MAX_BY(local_date, total_km) AS local_date, MAX_BY(route, total_km) AS detail FROM days
UNION ALL
SELECT 2, 'Top speed', ROUND(MAX(speed_kmh), 1), 'km/h', MAX_BY(local_date, speed_kmh), 'Recorded by the watch' FROM track
UNION ALL
SELECT 3, 'Highest point', ROUND(MAX(altitude_m), 0), 'm', MAX_BY(local_date, altitude_m), 'Watch altitude' FROM track
UNION ALL
SELECT 4, 'Lowest point', ROUND(MIN(altitude_m), 0), 'm', MIN_BY(local_date, altitude_m), 'Watch altitude, below sea level' FROM track
UNION ALL
SELECT 5, 'Furthest north', ROUND(MAX(lat), 3), '° latitude', MAX_BY(local_date, lat), 'Recorded position' FROM track
UNION ALL
SELECT 6, 'Twistiest day', MAX(curvature_deg_per_km), '°/km', MAX_BY(local_date, curvature_deg_per_km), MAX_BY(route, curvature_deg_per_km) FROM days
UNION ALL
SELECT 7, 'Biggest climbing day', MAX(recorded_ascent_m), 'm', MAX_BY(local_date, recorded_ascent_m), MAX_BY(route, recorded_ascent_m) FROM days
UNION ALL
SELECT 8, 'Slowest riding day', MIN(recorded_moving_avg_kmh), 'km/h average', MIN_BY(local_date, recorded_moving_avg_kmh), MIN_BY(route, recorded_moving_avg_kmh)
FROM days WHERE moving_h >= 1 UNION ALL
SELECT 9, 'Most estimated day', MAX(estimated_km), 'km', MAX_BY(local_date, estimated_km), MAX_BY(route, estimated_km) FROM days;

## Step 7: Day profiles for the app (app_profile)

The trip app draws a speed-and-altitude profile for a chosen day. Per-second data is far more than a phone chart needs, so this view buckets the track every **250 m** (`FLOOR(trip_km * 4)` — four buckets per km). Two extras:

* `day_km` subtracts the day's first bucket, so each day's profile starts at zero no matter where the day falls in the trip;
* `ts_adelaide` keeps local time, so the app can show a clock time without doing timezone maths.

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.trip.app_profile AS
WITH buckets AS (
  SELECT
    CAST(FLOOR(trip_km * 4) AS INT) AS bucket,
    MIN(trip_km) AS trip_km,
    MIN_BY(local_date, ts_utc) AS local_date,
    MIN(ts_adelaide) AS ts_adelaide,
    AVG(speed_kmh) AS speed_kmh,
    AVG(altitude_m) AS altitude_m,
    MAX(CAST(is_imputed AS INT)) = 1 AS is_imputed
  FROM workspace.trip.gold_track_filled
  GROUP BY CAST(FLOOR(trip_km * 4) AS INT)
)
SELECT
  local_date,
  ts_adelaide,
  ROUND(trip_km, 2) AS trip_km,
  ROUND(trip_km - MIN(trip_km) OVER (PARTITION BY local_date), 2) AS day_km,
  ROUND(speed_kmh, 1) AS speed_kmh,
  ROUND(altitude_m, 0) AS altitude_m,
  is_imputed,
  CASE WHEN is_imputed THEN 'Estimated' ELSE 'Recorded' END AS source
FROM buckets;

In [0]:
%sql
SELECT local_date, MIN(day_km) AS first_km, ROUND(MAX(day_km)) AS last_km
FROM workspace.trip.app_profile
GROUP BY local_date
ORDER BY local_date;

## Step 8: Check the results

Same habit as the pipeline's sanity cell: a handful of known-good expected values against what the views actually return. If `Total km` disagrees, the most likely cause is ordering — the pipeline was rebuilt after these expectations were written. The fix is always the same: run the pipeline notebook top to bottom, then this one, and update the expected values if the trip data has genuinely changed.

In [0]:
%sql
SELECT 'Total km' AS check_name, 4656 AS expected, total_km AS actual FROM workspace.trip.dash_kpis
UNION ALL SELECT 'Days', 15, COUNT(*) FROM workspace.trip.dash_daily
UNION ALL SELECT 'Speed band share, %', 100, ROUND(SUM(pct_of_riding_time), 0) FROM workspace.trip.dash_speed_bands
UNION ALL SELECT 'Records', 9, COUNT(*) FROM workspace.trip.dash_records;